In [22]:
pip install langchain langchain-community langgraph Chroma pypdf langchain-groq chromadb

In [29]:
from langgraph.graph import StateGraph , START , END
from google.colab import  userdata
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode , tools_condition
from typing import TypedDict , Annotated
from langchain_community.vectorstores import Chroma
from langchain_core.messages import HumanMessage, AIMessage ,  BaseMessage
from langgraph.graph.message import add_messages

In [2]:
api_key = userdata.get("groq_api_key")
model = ChatGroq(api_key=api_key , model="openai/gpt-oss-20b")

In [86]:
# Loaded the Document
pdf_loader = PyPDFLoader("/content/Huzaifa_CV.pdf")
docs = pdf_loader.load()

In [87]:
print(docs)

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-08-17T10:41:39+00:00', 'author': 'Un-named', 'moddate': '2026-08-17T10:41:39+00:00', 'source': '/content/Huzaifa_CV.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content="MOHAMMAD HUZAIFA  \nAgentic AI Developer  |  Full-Stack Engineer  |  LLM Systems  \n📧 [huzaifaqazi63@email.com]   |   📞 [0318-2425623]   |   📍 Karachi, Pakistan   |   github.com/MohammadHuzaifa-qazi  \nPROFESSIONAL SUMMARY  \n \nBSAI undergraduate (5th Semester) and Governor's Initiative scholar with 1+ year of hands-on experience building Agentic \nAI systems, LLM-powered applications, and production-ready full-stack web apps. Delivered 4 complete projects including 2 \nhackathon submissions and a live friend client website. Proficient in Python, TypeScript, Next.js, LangChain, LangGraph, and \nthe OpenAI Agents SDK. Seeking a software or AI internship to contribute to real-world products and gro

In [91]:
# Splitt the document
splitter = RecursiveCharacterTextSplitter(chunk_size=1000 , chunk_overlap=200)
chunks = splitter.split_documents(docs)
len(chunks)

7

In [92]:
# converts chunks text into embeddings(vector store representation)
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(
    chunks,
    embedding,
    persist_directory="chroma_database"
    )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [101]:
# retrieve the relevent search
retrive = vector_store.as_retriever(search_type="similarity" , search_kwargs={"k":2})

In [100]:
@tool
def rag_tool(question):
  """
  Retrieve relevant information from the pdf document.
  Use this tool when the user asks about a CV related question.
  that might be answered from the stored documents.
  """
  result = retrive.invoke(question)

  context = [docs.page_content for docs in result]
  metadata = [docs.metadata for docs in result]
  return{
      "question":question,
      "context":context,
      "metadata":metadata
  }


In [102]:
tools = [rag_tool]
workflow_tool = model.bind_tools(tools)

In [103]:
class rag_state(TypedDict):
  messages:Annotated[list[BaseMessage], add_messages]

In [104]:
def chat_node_function(state:rag_state):
  messages = state["messages"]
  res = workflow_tool.invoke(messages)
  return {"messages":res}

tool_node = ToolNode(tools)

In [105]:
graph = StateGraph(rag_state)

graph.add_node("chat_node" , chat_node_function)
graph.add_node("tools" , tool_node)

graph.add_edge(START , "chat_node")
graph.add_conditional_edges("chat_node" , tools_condition)
graph.add_edge("tools" , "chat_node")

workflow = graph.compile()

In [107]:
intial_state = {"messages":[HumanMessage(content="what is the skills of Huzaifa??")]}

output = workflow.invoke(intial_state)
print(output['messages'][-1].content)

**Huzaifa’s key technical skills (as highlighted in his CV)**  

| Category | Specific skills/tools |
|----------|-----------------------|
| **Programming languages** | Python, TypeScript |
| **Web development** | Next.js (React‑based framework) |
| **AI / LLM tooling** | LangChain, LangGraph, OpenAI Agents SDK |
| **Full‑stack & production readiness** | Building production‑ready full‑stack web apps, deploying and maintaining them |
| **Domain focus** | Agentic AI systems, LLM‑powered applications, Web 3.0 & Metaverse development (certification) |

These competencies are supported by his practical experience: 1+ year of hands‑on work building Agentic AI systems, completing 4 full projects (including hackathon submissions and a live client website).
